## 1.	CERNER LE PROBLÈME

Ce jeu de données (https://www.kaggle.com/datasets/shivamb/elevator-predictive-maintenance-dataset) est le fruit du travail de Cristian Axenie et Stefano Bortoli (https://zenodo.org/record/3653909) issu de la collecte des données de divers capteurs IoT et est destiné à la maintenance prédictive des portes d'ascenseur. Ces données permettent d'optimiser la maintenance prédictive des portes d'ascenseur afin de réduire les arrêts imprévus et de maximiser la durée de vie des équipements.

Le jeu de données contient des données d'exploitation, sous forme de séries temporelles échantillonnées à 4 Hz (4 fois par seconde), lors des pics d'utilisation et des heures creuses d'un ascenseur dans un immeuble (entre 16h30 et 23h30). Pour une porte de cabine, le système prend en compte : des capteurs électromécaniques (capteur de roulement à billes de porte), l'environnement (humidité) et la physique (vibrations).

Le problème qui est posé peut se résumer à la prédiction des pannes d'ascenseur. Notre objectif est de prédire (regression linéaire multiple) la valeur absolue des vibrations (le mouvement considéré toujours positif de la pièce par rapport au sol) afin de pouvoir classer (Arbre de décision) l'état de la pièce en fonction des 4 niveaux de sévérité des normes ISO10816 OU ISO20816. 


## 2.	RÉCUPÉRER LES DONNÉES
Ce jeu de données est disponible sur kaggle à l'adresse https://www.kaggle.com/datasets/shivamb/elevator-predictive-maintenance-dataset et sur zenodo à l'adresse https://zenodo.org/record/3653909. Nous avons importer ce jeu de données dans notre notebook.

In [ ]:
import pandas as pd
import numpy as np
df  = pd.read_csv("predictive-maintenance-dataset.csv")


### 2.1 Examiner la structure des données

In [ ]:
df.head(10)

In [ ]:
df.info()

Il y a 112001 lignes dans le tableau ce qui signifie qui est suffisant comme jeu d'apprentissage.

Noter que la variable cible vibration n'a que 109563 valeurs non null, il y a donc 2438 valeur manquante. Ce qui représente 2.2% du dataset, on peut donc supprimer ces valeurs sans risque.

In [ ]:
(112001-109563)/112001*100

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.describe()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
df.hist(bins=50,figsize=(20,15))
plt.title("Représentation graphique de chacune des variables")
plt.show()

La vibration minimale est 2, la vibration maximale est 100. Ce qui soulève des questions si on considère que l'unité de mesure des vibrations est en m/s2. Au seuil de vibration de 100 on aurait de manière certaine une destruction mécanique des pièces on peut donc se demander si l'on n'est pas ne présence de valeurs aberrantes.
Les échelles des colonnes x4 et x5 sont nettement plus grandes que les autres on pourrait être amené à les normaliser ou les standardiser par la suite.
On remarque une grande simulitude dans la distributin des varibles x1, x2 et révolution

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Crée des sous-graphiques (subplots) pour chaque colonne
df.plot(kind='box', subplots=True, layout=(4, 4), figsize=(20, 15), sharex=False, sharey=False)

plt.title("Boîte à moustache du jeu de données")
plt.tight_layout()
plt.show()


La présence des valeurs aberrantes ou outliers est confirmé dans les variables humidity, vibration et x5 grâce aux boites à moustaches

### 2.2 Création d'un jeu de test et d'un jeu d'entrainement

In [ ]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size=0.2, random_state=40)


In [ ]:
test_set.head()

In [ ]:
train_set.head()

## 3.	DÉCOUVRIR ET VISUALISER LES DONNÉES POUR MIEUX COMPRENDRE

Par la suite nous explorerons le jeu d'entrainement train_set pour éviter le biais d'espionnge.

In [ ]:
maintenance_df = train_set.copy()

### 3.1 Visualisation des données

In [ ]:
maintenance_df.columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calcul des statistiques
mean_vibration = maintenance_df["vibration"].mean()
median_vibration = maintenance_df["vibration"].median()

# Histogramme
sns.histplot(
    data=maintenance_df,
    x="vibration",
    kde=True,
    alpha=0.5,
)

# Ligne moyenne
plt.axvline(
    mean_vibration,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Moyenne = {mean_vibration:.2f}"
)

# Ligne médiane
plt.axvline(
    median_vibration,
    color="green",
    linestyle="-.",
    linewidth=2,
    label=f"Médiane = {median_vibration:.2f}"
)

# Titre et légende
plt.title("Distribution de la vibration")
plt.legend()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calcul des statistiques
mean_humidity = maintenance_df["humidity"].mean()
median_humidity = maintenance_df["humidity"].median()

# Histogramme
sns.histplot(
    data=maintenance_df,
    x="humidity",
    kde=True,
    alpha=0.5,
)

# Ligne moyenne
plt.axvline(
    mean_humidity,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Moyenne = {mean_humidity:.2f}"
)

# Ligne médiane
plt.axvline(
    median_humidity,
    color="green",
    linestyle="-.",
    linewidth=2,
    label=f"Médiane = {median_humidity:.2f}"
)

# Titre et légende
plt.title("Distribution de l'humidité")
plt.legend()

plt.show()

Au vu de l'écart entre la moyenne et la médiane on peut négliger les valeurs aberrantes contenue dans la variable humidity.

In [ ]:
#from pandas.plotting import scatter_matrix
#columns = ["humidity","vibration","x1","x2","x3","x4","x5"]
#columns = ["revolutions","humidity","vibration"]
#scatter_matrix(
#    maintenance_df[columns],
#    figsize=(12,8)#,
    #alpha=0.5,
    #diagonale="kde",
    #c=maintenance_df["vibration"],
    #cmap="viridis"
#    )
#plt.show()

### 3.2 Recherche des corrélations

In [ ]:
#corr_matrix = maintenance_df.corr()
#columns = ["revolutions","humidity","vibration","x1","x2","x3","x4","x5"]
corr_matrix = maintenance_df.corr().style.background_gradient(cmap='coolwarm')

corr_matrix

Nous éliminerons les variables "x1","x2","x3","x4","ID", qui sont dépendantes de la variable "revolutions", et
    "x5" qui est dépendante de la variable "humidity"

In [ ]:
corr_matrix = maintenance_df[["revolutions","humidity","vibration"]].corr().style.background_gradient(cmap='coolwarm')

corr_matrix

In [ ]:
maintenance_df[["revolutions","humidity","vibration"]].skew()

La coefficient d'asymétrie skewness nous montre un fort étalement vers la droite (0.983) pour la variable cible vibration ce qui indique la présence forte problable de valeurs aberrantes.

## 4.	PRÉPARER LES DONNÉES POUR LES ALGORITHMES DE ML

## 5.	SÉLECTIONNER ET ENTRAÎNER UN MODÈLE

## 6.	RÉGLER AVEC PRÉCISION VOTRE MODÈLE